# IDD Segmentation -> YOLO Segmentation Pipeline
This notebook performs the full pipeline: subset selection, JSON-to-YOLO conversion, training a lightweight YOLOv8-seg model, and inference with drivable area post-processing.


## 1. Environment Setup & Check GPU


In [ ]:
!pip install ultralytics opencv-python matplotlib
import ultralytics
ultralytics.checks()
!nvidia-smi


## 2. Mount Google Drive


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Not running in Colab, or drive already mounted.")


## 3. Configuration


In [ ]:
import os
import glob
import json
import shutil
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Configurables
DATASET_ROOT = '/content/drive/MyDrive/IDD_Segmentation' # Change this to your actual dataset path in Google Drive
OUTPUT_ROOT = '/content/IDD_YOLO_SEG'
MODEL_NAME = 'yolov8n-seg.pt'
IMAGE_SIZE = 640
EPOCHS = 30
BATCH_SIZE = 16
TRAIN_SUBSET_SIZE = 1500
VAL_SUBSET_SIZE = 300

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("Configuration set.")


## 4. Class Mapping
We create a CSV mapping to map the 39 fine-grained IDD classes to the 18 target project classes.


In [ ]:
mapping_csv = """original_class,target_class_id,target_class_name,reason
road,8,road,Direct mapping to project class
sky,-1,ignore,Background/sky not needed for path planning
vegetation,-1,ignore,Background vegetation not needed
building,-1,ignore,Background structures not needed
drivable fallback,9,drivable_fallback,Direct mapping to project class for unpaved roads
obs-str-bar-fallback,17,obs_str_bar_fallback,Important static boundary obstacle
car,0,car,Direct mapping to project class
wall,13,wall,Important static boundary
non-drivable fallback,12,non_drivable_fallback,Direct mapping for non-drivable shoulders
truck,2,truck,Direct mapping to project class
billboard,-1,ignore,Static signage usually elevated and not an obstacle
sidewalk,10,sidewalk,Direct mapping to project class
curb,11,curb,Direct mapping to project class
bus,1,bus,Direct mapping to project class
motorcycle,4,two_wheeler,Mapped to general two-wheeler category
pole,16,pole,Important static vertical obstacle
autorickshaw,3,autorickshaw,Direct mapping to project class
bridge,-1,ignore,Elevated infrastructure not an immediate obstacle
fence,14,fence,Important static boundary
rider,6,pedestrian,Riders are vulnerable humans sharing road space
person,6,pedestrian,Direct mapping to project class
vehicle fallback,0,car,Fallback vehicles grouped into general car obstacle category
guard rail,15,guard_rail,Important static boundary
traffic sign,-1,ignore,Elevated signage not a physical obstacle
fallback background,-1,ignore,Background noise
bicycle,5,bicycle,Direct mapping to project class
caravan,2,truck,Caravans/trailers grouped with large truck obstacles
polegroup,16,pole,Groups of poles mapped to pole class
animal,7,animal,Direct mapping to project class for cattle/stray dogs
parking,-1,ignore,Typically not part of primary active path planning
traffic light,-1,ignore,Elevated signal not a physical obstacle
out of roi,-1,ignore,Dataset artifact
rail track,-1,ignore,Not a physical 3D obstacle
trailer,2,truck,Trailers grouped with large truck obstacles
tunnel,-1,ignore,Not an immediate collision obstacle
train,-1,ignore,Not a standard road obstacle
license plate,-1,ignore,Vehicle sub-part
rectification border,-1,ignore,Dataset artifact
ego vehicle,-1,ignore,Host vehicle body
"""

with open(os.path.join(OUTPUT_ROOT, 'segmentation_class_mapping.csv'), 'w') as f:
    f.write(mapping_csv)

# Parse mapping
class_mapping = {}
target_classes = {}
lines = mapping_csv.strip().split('\n')[1:]
for line in lines:
    orig, tid, tname, _ = line.split(',')
    tid = int(tid)
    class_mapping[orig] = tid
    if tid != -1:
        target_classes[tid] = tname

print(f"Loaded {len(class_mapping)} mappings. Target classes: {len(target_classes)}")



## 5. Select Subset
Select images uniformly across sequence directories to maintain diversity.


In [ ]:
def select_subset(split, target_size):
    img_dir = os.path.join(DATASET_ROOT, 'leftImg8bit', split)
    if not os.path.exists(img_dir):
        print(f"Path not found: {img_dir}. Please ensure DATASET_ROOT is correct.")
        return []
        
    seq_folders = glob.glob(os.path.join(img_dir, '*'))
    if not seq_folders:
        print(f"No sequence folders found in {img_dir}")
        return []
        
    images_per_seq = target_size // len(seq_folders)
    selected = []
    
    for seq in seq_folders:
        imgs = sorted(glob.glob(os.path.join(seq, '*.png')))
        if len(imgs) > images_per_seq:
            sampled = random.sample(imgs, images_per_seq)
        else:
            sampled = imgs
        selected.extend(sampled)
        
    # If we need more due to rounding
    while len(selected) < target_size:
        all_remaining = set(glob.glob(os.path.join(img_dir, '*/*.png'))) - set(selected)
        if not all_remaining:
            break
        selected.append(random.sample(list(all_remaining), 1)[0])
        
    return selected[:target_size]

train_images = select_subset('train', TRAIN_SUBSET_SIZE)
val_images = select_subset('val', VAL_SUBSET_SIZE)

print(f"Selected {len(train_images)} train images and {len(val_images)} val images.")



## 6. Convert JSON to YOLO Segmentation Format


In [ ]:
def convert_to_yolo(images_list, split):
    out_img_dir = os.path.join(OUTPUT_ROOT, 'images', split)
    out_lbl_dir = os.path.join(OUTPUT_ROOT, 'labels', split)
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)
    
    valid_count = 0
    for img_path in images_list:
        # Resolve annotation path
        # Assuming IDD format: leftImg8bit/train/seq/id_leftImg8bit.png -> gtFine/train/seq/id_gtFine_polygons.json
        base_name = os.path.basename(img_path).replace('_leftImg8bit.png', '')
        seq_name = os.path.basename(os.path.dirname(img_path))
        anno_path = os.path.join(DATASET_ROOT, 'gtFine', split, seq_name, f"{base_name}_gtFine_polygons.json")
        
        if not os.path.exists(anno_path):
            continue
            
        with open(anno_path, 'r') as f:
            data = json.load(f)
            
        img_w = data['imgWidth']
        img_h = data['imgHeight']
        
        yolo_lines = []
        for obj in data['objects']:
            label = obj['label']
            if label not in class_mapping:
                continue
            
            tid = class_mapping[label]
            if tid == -1:
                continue # ignore class
                
            poly = obj['polygon']
            if len(poly) < 3:
                continue # invalid polygon
                
            # Normalize
            normalized_poly = []
            valid_poly = True
            for point in poly:
                nx = max(0.0, min(1.0, point[0] / img_w))
                ny = max(0.0, min(1.0, point[1] / img_h))
                normalized_poly.extend([nx, ny])
                
            if valid_poly:
                line = f"{tid} " + " ".join([f"{v:.6f}" for v in normalized_poly])
                yolo_lines.append(line)
                
        if yolo_lines:
            # Save label
            lbl_path = os.path.join(out_lbl_dir, f"{base_name}.txt")
            with open(lbl_path, 'w') as f:
                f.write("\n".join(yolo_lines))
            # Copy image
            dest_img = os.path.join(out_img_dir, f"{base_name}.png")
            shutil.copy(img_path, dest_img)
            valid_count += 1
            
    print(f"Processed {valid_count} images for {split}")

print("Converting training set...")
convert_to_yolo(train_images, 'train')
print("Converting validation set...")
convert_to_yolo(val_images, 'val')



## 7. Create data.yaml


In [ ]:
yaml_content = f"""path: {OUTPUT_ROOT}
train: images/train
val: images/val

names:
"""
for tid in sorted(target_classes.keys()):
    yaml_content += f"  {tid}: {target_classes[tid]}\n"

yaml_path = os.path.join(OUTPUT_ROOT, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(yaml_content)



## 8. Visualize Conversion Example


In [ ]:
def visualize_yolo_annotation(image_path, label_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    colors = plt.cm.get_cmap('hsv', len(target_classes))
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                cid = int(parts[0])
                poly_norm = np.array(parts[1:], dtype=float).reshape(-1, 2)
                poly_abs = (poly_norm * [w, h]).astype(np.int32)
                
                color = [int(c*255) for c in colors(cid)[:3]]
                cv2.polylines(img, [poly_abs], isClosed=True, color=color, thickness=2)
                cv2.putText(img, target_classes[cid], tuple(poly_abs[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
                
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

sample_imgs = glob.glob(os.path.join(OUTPUT_ROOT, 'images/train/*.png'))
if sample_imgs:
    sample_img = sample_imgs[0]
    sample_lbl = sample_img.replace('images', 'labels').replace('.png', '.txt')
    visualize_yolo_annotation(sample_img, sample_lbl)



## 9. Train YOLO Segmentation Model


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME) # Load pre-trained lightweight segmentation model

results = model.train(
    data=os.path.join(OUTPUT_ROOT, 'data.yaml'),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    name='idd_seg_model',
    patience=10, # early stopping
    device=0 # assumes GPU is available
)



## 10. Evaluate Model


In [ ]:
metrics = model.val()
print("mAP50-95 (Mask):", metrics.seg.map)
print("mAP50 (Mask):", metrics.seg.map50)



## 11. Inference & Drivable Area Function


In [ ]:
def predict_image(image_path, model):
    results = model.predict(image_path, imgsz=IMAGE_SIZE, conf=0.25)
    result = results[0]
    
    structured_output = []
    
    img = cv2.imread(image_path)
    img_h, img_w = img.shape[:2]
    
    drivable_mask = np.zeros((img_h, img_w), dtype=np.uint8)
    non_drivable_mask = np.zeros((img_h, img_w), dtype=np.uint8)
    
    if result.masks is not None:
        for box, mask in zip(result.boxes, result.masks):
            cid = int(box.cls[0])
            conf = float(box.conf[0])
            cname = target_classes.get(cid, "unknown")
            
            # Box
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            
            # Mask polygon (normalized -> absolute)
            poly_norm = mask.xyn[0]
            poly_abs = (poly_norm * [img_w, img_h]).astype(np.int32)
            
            structured_output.append({
                "class_id": cid,
                "class_name": cname,
                "confidence": conf,
                "polygon": poly_abs.tolist(),
                "polygon_normalized": poly_norm.tolist(),
                "bbox": {"x1": x1, "y1": y1, "x2": x2, "y2": y2}
            })
            
            # Compute derived regions
            if cname in ['road', 'drivable_fallback']:
                cv2.fillPoly(drivable_mask, [poly_abs], 1)
            else:
                cv2.fillPoly(non_drivable_mask, [poly_abs], 1)
                
    # Drivable area visualization
    vis_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Overlay green for drivable, red for non-drivable
    green_overlay = np.zeros_like(vis_img)
    green_overlay[drivable_mask == 1] = [0, 255, 0]
    
    red_overlay = np.zeros_like(vis_img)
    red_overlay[non_drivable_mask == 1] = [255, 0, 0]
    
    vis_img = cv2.addWeighted(vis_img, 1.0, green_overlay, 0.4, 0)
    vis_img = cv2.addWeighted(vis_img, 1.0, red_overlay, 0.4, 0)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(vis_img)
    plt.title("Drivable Area Visualization")
    plt.axis('off')
    plt.show()
    
    return structured_output

# Test inference on a validation image
if glob.glob(os.path.join(OUTPUT_ROOT, 'images/val/*.png')):
    test_img = glob.glob(os.path.join(OUTPUT_ROOT, 'images/val/*.png'))[0]
    preds = predict_image(test_img, model)
    print("Example Prediction:", json.dumps(preds[0], indent=2) if preds else "No predictions")



## 12. Final Report


In [ ]:
print("""
==================================================
FINAL REPORT
==================================================
Dataset: IDD Segmentation (IDD 20k Part I)
Number of images (Target subset): 1800
Number of training images: 1500
Number of validation images: 300
Number of target classes: 18

Model:
Architecture: YOLOv8-seg (nano/small)
Image size: {IMAGE_SIZE}
Epochs: {EPOCHS}
Batch size: {BATCH_SIZE}

Model location: runs/segment/idd_seg_model/weights/best.pt
""")



## 13. Export Model


In [ ]:
# Save best model to ONNX for deployment
success = model.export(format='onnx')
print(f"Model exported to ONNX: {success}")

